# RNN Architecture Comparison
**Winner: Vanilla RNN (0.2175) | Worst: BiLSTM (0.2312)**

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

# Data and utilities
from utils.data_loadingV2 import load_data
from utils.band_mapping import load_band_mapping
from utils.training import train_two_stage, get_predictions
from utils.reproducibility_utils import seed_everything
from utils.visualization import plot_method_comparison, plot_predictions_scatter

# Models from models/ folder
from models.vanilla_rnn import VanillaRNN
from models.simple_lstm import SimpleLSTM
from models.simple_gru import SimpleGRU
from models.simple_bilstm import SimpleBiLSTM
from models.bilstm_attention import BiLSTMAttention

SEED = 42
seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Project root: {PROJECT_ROOT}")

Device: cuda
Project root: /home/mdesouky/workspace/mait_g2_drift_challenge


In [2]:
# Load data
TRAIN_PATH = PROJECT_ROOT / "datasets_challenge" / "train"
COLUMNS_TO_DROP = ['sideSlip', 'vyCG', 'longitude', 'latitude', 'time']

train_loader, val_loader, scaler, _, _ = load_data(
    train_path=str(TRAIN_PATH), val_ratio=0.15, columns_to_drop=COLUMNS_TO_DROP,
    sequence_length=20, target="sideSlip", threshold=True, threshold_value=3.0,
    threshold_column="vxCG", batchsize=64, scaling=True, seed=SEED, shuffle=True
)
band_dict = load_band_mapping()
print(f"Train: {len(train_loader)} batches, Val: {len(val_loader)} batches")

V5 Data Loader: Stratified Temporal Split

Processing 22 files...
Total groups after threshold: 42

Train: Many=90.3%, Medium=7.6%, Few=2.1%
Val:   Many=96.6%, Medium=2.3%, Few=1.1%

Train: 120,445 sequences | Val: 16,635 sequences

Train: 1882 batches, Val: 260 batches


## Architecture Comparison

In [ ]:
architectures = [
    ('VanillaRNN', VanillaRNN(22, 64)),
    ('LSTM', SimpleLSTM(22, 64)),
    ('GRU', SimpleGRU(22, 64)),
    ('BiLSTM', SimpleBiLSTM(22, 64)),
    ('BiLSTM+Attention', BiLSTMAttention(22, 64, 2, 4)),
]

results = []
for name, model in architectures:
    seed_everything(SEED)
    params = sum(p.numel() for p in model.parameters())
    print(f"\nTraining {name} ({params:,} params)...")
    
    _, metrics = train_two_stage(model, train_loader, val_loader, band_dict, device)
    print(f"  Overall={metrics['overall_mae']:.4f}, Few={metrics['few_mae']:.4f}")
    results.append({'model': name, 'params': params, **metrics})

pd.DataFrame(results).sort_values('overall_mae')


Training VanillaRNN (9,857 params)...


## Predictions vs Ground Truth

In [ ]:
# Train winner
seed_everything(SEED)
winner = VanillaRNN(22, 64)
winner, _ = train_two_stage(winner, train_loader, val_loader, band_dict, device)

val_preds, val_targets = get_predictions(winner, val_loader, device)

# Using visualization from utils/visualization.py
fig = plot_predictions_scatter(val_preds, val_targets, title='Vanilla RNN: Predictions vs Ground Truth')
fig.show()

# Also show comparison of architectures
results_for_plot = [{'method': r['model'], 'score': r['overall_mae']} for r in results]
fig_compare = plot_method_comparison(results_for_plot, title='Architecture Comparison: Overall MAE')
fig_compare.show()

## Summary
| Finding | Result |
|---------|--------|
| Best architecture | Vanilla RNN (64h, 1L) |
| Bidirectional benefit | None (20 timesteps too short) |
| Attention benefit | Marginal |
| More params | Worse (overfitting) |

## Interactive Predictions vs Ground Truth

In [ ]:
# Interactive line plot showing predictions overlaid on ground truth
from utils.visualization import plot_predictions_vs_truth

# Using the winning VanillaRNN model predictions
fig_line = plot_predictions_vs_truth(
    val_preds, val_targets,
    title='Vanilla RNN: Predictions vs Ground Truth (Interactive)',
    max_samples=5000  # Subsample for performance
)
fig_line.show()

NameError: name 'val_preds' is not defined